In [1]:
import os
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/sales.csv'
    OUT_DIR = '/content/drive/MyDrive'
except ImportError:
    DATA_PATH = 'sales.csv'
    OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
def make_demo_sales(n=3636, seed=42):
    rng = np.random.default_rng(seed)
    prices = {'Latte': 38.7, 'Hot Chocolate': 38.7, 'Americano': 28.9,
              'Americano with Milk': 33.8, 'Cappuccino': 38.7, 'Cocoa': 35.76,
              'Cortado': 30.0, 'Espresso': 25.96}
    names = rng.choice(list(prices), size=n)
    start, end = pd.Timestamp('2024-03-01 10:00'), pd.Timestamp('2025-03-23 19:00')
    dt = pd.Series(pd.to_datetime(np.sort(rng.integers(start.value, end.value, size=n))))
    cash_type = np.where(rng.random(n) < 0.9, 'card', 'cash')
    card = [f'ANON-0000-0000-{i:04d}' if t == 'card' else None
            for t, i in zip(cash_type, rng.integers(1, 1200, size=n))]
    return pd.DataFrame({
        'date': dt.dt.strftime('%Y-%m-%d'),
        'datetime': dt.dt.strftime('%Y-%m-%d %H:%M:%S.%f').str[:-3],
        'cash_type': cash_type,
        'card': card,
        'money': [prices[x] for x in names],
        'coffee_name': names,
    })

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    print(f'Файл {DATA_PATH} не знайдено - використовую демонстраційні дані.')
    df = make_demo_sales()

df.head()

,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 11:58:48.618000,card,ANON-0000-0000-0450,33.80,Hot chocolate
1,2024-03-01,2024-03-01 13:43:22.828000,card,ANON-0000-0000-0207,28.90,Americano
2,2024-03-01,2024-03-01 14:33:30.639000,cash,NaN,35.76,Cocoa
3,2024-03-01,2024-03-01 14:52:46.593000,card,ANON-0000-0000-0484,33.80,Americano with milk
4,2024-03-01,2024-03-01 15:16:40.466000,card,ANON-0000-0000-0285,38.70,Latte


In [3]:
revenue_by_coffee = df.groupby('coffee_name')['money'].sum()
print(revenue_by_coffee)

coffee_name
Americano               6097.90
Americano with milk     3785.60
Cappuccino              5650.20
Cocoa                   2217.12
Cortado                 1450.00
Espresso                1632.00
Hot chocolate           2974.40
Latte                  10178.10
Name: money, dtype: float64


In [4]:
avg_money = df.groupby('coffee_name')['money'].mean()
print(avg_money)

coffee_name
Americano              28.90
Americano with milk    33.80
Cappuccino             38.70
Cocoa                  35.76
Cortado                29.00
Espresso               24.00
Hot chocolate          33.80
Latte                  38.70
Name: money, dtype: float64


In [5]:
trans_count = df.groupby(['date', 'coffee_name']).size()
print(trans_count)

date        coffee_name        
2024-03-01  Americano              3
            Americano with milk    1
            Cocoa                  1
            Cortado                1
            Espresso               1
                                  ..
2024-03-31  Americano with milk    2
            Cortado                1
            Espresso               1
            Hot chocolate          3
            Latte                  2
Length: 229, dtype: int64


In [6]:
trans_count_df = df.groupby(['date', 'coffee_name']).size().reset_index(name='count')
print(trans_count_df.head(10))

         date          coffee_name  count
0  2024-03-01            Americano      3
1  2024-03-01  Americano with milk      1
2  2024-03-01                Cocoa      1
3  2024-03-01              Cortado      1
4  2024-03-01             Espresso      1
5  2024-03-01        Hot chocolate      3
6  2024-03-01                Latte      2
7  2024-03-02            Americano      8
8  2024-03-02  Americano with milk      5
9  2024-03-02           Cappuccino      2


In [7]:
summary = df.groupby('coffee_name')['money'].agg(['sum', 'mean', 'min', 'max'])
print(summary)

                          sum   mean    min    max
coffee_name                                       
Americano             6097.90  28.90  28.90  28.90
Americano with milk   3785.60  33.80  33.80  33.80
Cappuccino            5650.20  38.70  38.70  38.70
Cocoa                 2217.12  35.76  35.76  35.76
Cortado               1450.00  29.00  29.00  29.00
Espresso              1632.00  24.00  24.00  24.00
Hot chocolate         2974.40  33.80  33.80  33.80
Latte                10178.10  38.70  38.70  38.70


In [8]:
df['day_of_week'] = pd.to_datetime(df['datetime']).dt.dayofweek

pivot = pd.pivot_table(df, values='money', index='coffee_name',
                       columns='day_of_week', aggfunc='sum', fill_value=0)
print(pivot)

day_of_week                0        1        2       3       4        5  \
coffee_name                                                               
Americano             953.70   780.30   693.60   722.5  1040.4  1127.10   
Americano with milk   743.60   507.00   676.00   540.8   439.4   338.00   
Cappuccino            735.30  1238.40   348.30   774.0   928.8   890.10   
Cocoa                 214.56   143.04   500.64   178.8   357.6   500.64   
Cortado               174.00   203.00   261.00   261.0   174.0   261.00   
Espresso              264.00   240.00   336.00   240.0   216.0   192.00   
Hot chocolate         405.60   439.40   473.20   338.0   540.8   405.60   
Latte                1199.70   967.50  1470.60  1625.4  1083.6  2167.20   

day_of_week                6  
coffee_name                   
Americano             780.30  
Americano with milk   540.80  
Cappuccino            735.30  
Cocoa                 321.84  
Cortado               116.00  
Espresso              144.00  
H

Колонка `day_of_week`: понеділок = 0, неділя = 6.

In [9]:
crosstab = pd.crosstab(df['coffee_name'], df['cash_type'])
print(crosstab)

cash_type            card  cash
coffee_name                    
Americano             183    28
Americano with milk   101    11
Cappuccino            134    12
Cocoa                  53     9
Cortado                41     9
Espresso               61     7
Hot chocolate          83     5
Latte                 247    16


In [10]:
categories = pd.DataFrame({
    'coffee_name': ['Latte', 'Americano', 'Hot Chocolate', 'Cappuccino'],
    'category': ['молочний', 'чорний', 'молочний', 'молочний']
})

df_merged = df.merge(categories, on='coffee_name', how='left')
print(df_merged.head())

         date                    datetime cash_type                 card  \
0  2024-03-01  2024-03-01 11:58:48.618000      card  ANON-0000-0000-0450   
1  2024-03-01  2024-03-01 13:43:22.828000      card  ANON-0000-0000-0207   
2  2024-03-01  2024-03-01 14:33:30.639000      cash                  NaN   
3  2024-03-01  2024-03-01 14:52:46.593000      card  ANON-0000-0000-0484   
4  2024-03-01  2024-03-01 15:16:40.466000      card  ANON-0000-0000-0285   

   money          coffee_name  day_of_week  category  
0  33.80        Hot chocolate            4       NaN  
1  28.90            Americano            4    чорний  
2  35.76                Cocoa            4       NaN  
3  33.80  Americano with milk            4       NaN  
4  38.70                Latte            4  молочний  


`how='left'` зберігає всі рядки `df`; для напоїв, яких немає в `categories`, у колонці `category` буде `NaN`.

In [11]:
df_part1 = df[df['date'] <= '2024-03-15']
df_part2 = df[df['date'] > '2024-03-15']

df_concat = pd.concat([df_part1, df_part2])
print(f"Розміри: {len(df_part1)} + {len(df_part2)} = {len(df_concat)}")

Розміри: 498 + 502 = 1000


In [12]:
left = pd.DataFrame({'A': [1, 2, 3]}, index=['a', 'b', 'c'])
right = pd.DataFrame({'B': [10, 20, 30]}, index=['a', 'b', 'c'])

joined = left.join(right)
print(joined)

   A   B
a  1  10
b  2  20
c  3  30


In [13]:
total_revenue = df['money'].sum()
revenue_share = (df.groupby('coffee_name')['money'].sum() / total_revenue * 100).round(2)
print(revenue_share)

coffee_name
Americano              17.94
Americano with milk    11.14
Cappuccino             16.63
Cocoa                   6.52
Cortado                 4.27
Espresso                4.80
Hot chocolate           8.75
Latte                  29.95
Name: money, dtype: float64


## Висновки
Виконано групування за однією та кількома колонками, агрегацію з `agg`, побудову зведеної та перехресної таблиць, об’єднання даних через `merge`, `concat` і `join`, а також розрахунок частки доходу кожного напою.